In [ ]:
import tomllib
import os
from dotenv import load_dotenv
from tqdm import tqdm
import glob

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

load_dotenv()

TEST_HAND_FILE = os.getenv('TEST_HAND_FILE')
PARQUET_DATA_DIR = os.getenv('PARQUET_DATA_DIR')

In [ ]:
def translate_card_names(card_str):
    """ Translates the shorthand from PHH into english card names. 
    expects 2 character card strings, e.g. 'Ah' for Ace of Hearts
    """
    rank_translation = {
        '2': 'Two',
        '3': 'Three',
        '4': 'Four',
        '5': 'Five',
        '6': 'Six',
        '7': 'Seven',
        '8': 'Eight',
        '9': 'Nine',
        'T': 'Ten',
        'J': 'Jack',
        'Q': 'Queen',
        'K': 'King',
        'A': 'Ace'
    }
    suit_translation = {
        'h': 'Hearts',
        'd': 'Diamonds',
        'c': 'Clubs',
        's': 'Spades'
    }
    rank = card_str[0]
    suit = card_str[1]
    return f"{rank_translation[rank]} of {suit_translation[suit]}"

In [ ]:
def actions_translation(action_str):
    """ 
    UNDER DEVLOPMENT!!!
    Translates the shorthand from PHH into english action names. 
    expects single character action strings, e.g. 'c' for call
    example:

    'p1 cbr 1.50' -> {'player': 'p1', 'action': 'raise', 'amount': 1.50}

    """
    for action in action_str:
        action = action.replace('[', '').replace(']', '') #Remove brackets from actions
        action = action.replace("'", "") #Remove spaces from actions
        action = action.strip() #Remove leading/trailing whitespace
        action_parts = action.split(' ')
        action_description = ''
        
        if action_parts[0] == 'd': #Dealer action:
            role = 'Dealer'
            if action_parts[1] == 'dh': #Dealing hole cards
                action = 'Deal Hole Cards'
                action_description = f'{role} deals hole cards to player {action_parts[2]}'
            elif action_parts[1] == 'db': #Dealing board cards
                action = 'Deal Board Cards'
                cards = action_parts[2]
                if len(cards) % 2 != 0:
                    raise ValueError(f"Invalid card string: {cards}")
                else:
                    card_list = [translate_card_names(cards[i:i+2]) for i in range(0, len(cards), 2)] #Check output of this to make sure it works as expected
                action_description = f'{role} deals board cards'
            else:
                action = 'Unknown Dealer Action'

        if action_parts[0].startswith('p'): #Player action:
            role = 'Player'
            player_id = action_parts[0][-1:] #Remove the 'p' from the player id
            if action_parts[1] == 'cc': #Call
                action = 'Calls'
            elif action_parts[1] == 'cbr': #Raise
                action = f'Raises to {action_parts[2]}'
            elif action_parts[1] == 'f': #Fold
                action = 'Folds'
            else:
                action = 'Unknown Player Action'
            action_description = f'{role} {player_id} {action}'

    return action_description

In [ ]:
def parse_actions(actions_list, hand_id):
    """ Parses the list of actions from PHH into a structured format. 
    expects a list of action strings, e.g. ['p1 cbr 1.50', 'd db AhKhQh']
    output structured format example:
[
    {'hand_id': '13', 'action_seq': 1, 'street': 'preflop', 'player': 1, 'action': 'raise', 'amount': 1.50},
]
    I am only capturing player actions.
    Hand Id is required to link the actions to the correct hand.
    I also need a sequence number for each action to maintain the order of actions within a hand.
    """
    parsed_actions = []
    street = 0 #0 = preflop, 1 = flop, 2 = turn, 3 = river
    for action_str in actions_list:
        parsed_action = {}
        parsed_action['hand_id'] = str(hand_id)
        
        if action_str.startswith('d') and 'db' in action_str: #Dealer action (ignore dealer hole card actions):
            street += 1
            continue #Skip dealer actions for now, we can come back to them later if needed
        
        if action_str.startswith('p'): #Player action:
            parsed_action['player'] = int(action_str.split(' ')[0][1:]) #Extract player id from action string
            #map action verbs:
            verb = action_str.split(' ')[1].strip()
            if verb == 'cc':
                parsed_action['action'] = 'call'
                parsed_action['amount'] = None #Amount is not specified for calls in PHH, we can calculate it later if needed
            elif verb == 'cbr':
                parsed_action['action'] = 'raise'
                parsed_action['amount'] = float(action_str.split(' ')[2]) #Extract raise amount from action string
            elif verb == 'f':
                parsed_action['action'] = 'fold'
                parsed_action['amount'] = None #No amount for folds
            elif verb == 'sm':
                if '????' in action_str:
                    parsed_action['action'] = 'muck'
                else:
                    parsed_action['action'] = 'show'
                parsed_action['amount'] = None #No amount for show/muck actions
            else:
                parsed_action['action'] = 'unknown'
                parsed_action['amount'] = None
                print(f"Warning: Unrecognized action verb '{verb}' in action string '{action_str}'")
        
        if parsed_action.get('player', None) is not None: #Only append player actions for now
            parsed_action['action_seq'] = len(parsed_actions) + 1
            parsed_action['street'] = ['preflop', 'flop', 'turn', 'river'][street]
            parsed_actions.append(parsed_action)
    return parsed_actions

In [ ]:
def parse_hand(single_hand_dict):
    """ Parses a single hand dictionary from PHH into structured formats for hand, player, and actions. 
    expects a single hand dictionary from PHH, e.g. data['13']
    output structured formats:
    hand_dict = {'id': '13', 'stakes': '50NLH', 'board': ['Ah', 'Kh', 'Qh']}
    player_dict = {
        1: {'seat': 1, 'position': 1, 'starting_stack': 100.0, 'amount_won_lost': -10.0},
        2: {'seat': 2, 'position': 2, 'starting_stack': 100.0, 'amount_won_lost': 10.0},
    }
    actions_dict = [
        {'hand_id': '13', 'action_seq': 1, 'street': 'preflop', 'player': 1, 'action': 'raise', 'amount': 1.50},
        {'hand_id': '13', 'action_seq': 2, 'street': 'preflop', 'player': 2, 'action': 'call', 'amount': None},
        ...
    ]
    """
    hand_dict = {} #id, stakes, board cards
    player_dict = {} #one row per player: seat, position, starting stack, amount won/lost
    actions_dict = {} #one row per action: street, player, action type, amount

    # Hand Dict
    hand_dict['id'] = str(single_hand_dict['hand'])
    hand_dict['stakes'] = single_hand_dict['blinds_or_straddles']
    # Assumption: The first two entries in the blinds_or_straddles list are always the small blind and big blind, respectively. This is based on the structure of the PHH data, but should be verified with additional samples to ensure consistency.
    hand_dict['small_blind'] = single_hand_dict['blinds_or_straddles'][0]
    hand_dict['big_blind'] = single_hand_dict['blinds_or_straddles'][1]

    board_cards = [x.replace('d db', '').strip() for x in single_hand_dict['actions'] if 'db' in x]
    hand_dict['board'] = board_cards

    # Player Dict
    for idx, current_player in enumerate(single_hand_dict['players']):
        player_dict[current_player] = {
            'seat': single_hand_dict['seats'][idx],
            'position':  idx + 1,
            'starting_stack': single_hand_dict['starting_stacks'][idx],
            'amount_won_lost': single_hand_dict.get('winnings', [0] * len(single_hand_dict['players'])) [idx],
        }


    # Actions Dict
    actions_dict = parse_actions(single_hand_dict['actions'], single_hand_dict['hand'])
    return hand_dict, player_dict, actions_dict

In [ ]:
def clear_parquet_files():
    """ Utility function to clear existing parquet files before writing new data. """
    if os.path.exists(PARQUET_DATA_DIR):
        files = glob.glob(pathname=os.path.join(PARQUET_DATA_DIR, "**/*.parquet"), root_dir=PARQUET_DATA_DIR, recursive=True)
        for file in files:
            try:
                if os.path.isfile(file):
                    os.unlink(file)
            except Exception as e:
                print(f"Error deleting {file_path}: {e}")
    else:
        os.makedirs(PARQUET_DATA_DIR)

In [ ]:
# Declaring Schema for Parquet files --- UNDER DEVLOPMENT!!!

actions_schema = pa.schema([
    pa.field('hand_id',    pa.string(),  nullable=True),
    pa.field('player',     pa.int8(),    nullable=True),
    pa.field('action',     pa.string(),  nullable=True),
    pa.field('amount',     pa.float64(), nullable=True),
    pa.field('action_seq', pa.int16(),   nullable=True),
    pa.field('street',     pa.string(),  nullable=True),
])

players_schema = pa.schema([
    pa.field('hand_id',       pa.string(),  nullable=True),
    pa.field('player_name',   pa.string(),  nullable=True),
    pa.field('seat',          pa.int8(),    nullable=True),
    pa.field('position',      pa.int8(),    nullable=True),
    pa.field('starting_stack',pa.float64(), nullable=True),
    pa.field('amount_won_lost', pa.float64(), nullable=True),
])

hands_schema = pa.schema([
    pa.field('id',          pa.string(),  nullable=True),
    pa.field('stakes',      pa.list_(pa.float64()),  nullable=True),
    pa.field('small_blind', pa.float64(), nullable=True),
    pa.field('big_blind',   pa.float64(), nullable=True),
    pa.field('board',       pa.list_(pa.string()), nullable=True),
])

In [ ]:
def parquet_write_batch(hands, players, actions):
    
    Hand_Part_ID = len(os.listdir(f'{PARQUET_DATA_DIR}/Hands')) + 1
    hands_df = pd.DataFrame(hands)
    table_hands = pa.Table.from_pandas(hands_df, schema=hands_schema)

    Players_Part_ID = len(os.listdir(f'{PARQUET_DATA_DIR}/Players')) + 1
    players_flat = [
        {'hand_id': hands[i]['id'], 'player_name': name, **stats}
        for i, player_dict in enumerate(players)
        for name, stats in player_dict.items()
    ]
    players_df = pd.DataFrame(players_flat)
    table_players = pa.Table.from_pandas(players_df, schema=players_schema)

    Actions_Part_ID = len(os.listdir(f'{PARQUET_DATA_DIR}/Actions')) + 1
    actions_df = pd.DataFrame(actions)
    table_actions = pa.Table.from_pandas(actions_df, schema=actions_schema)

    pq.write_table(table_hands, f'{PARQUET_DATA_DIR}/Hands/hands_{Hand_Part_ID}.parquet')
    pq.write_table(table_players, f'{PARQUET_DATA_DIR}/Players/players_{Players_Part_ID}.parquet')
    pq.write_table(table_actions, f'{PARQUET_DATA_DIR}/Actions/actions_{Actions_Part_ID}.parquet')

In [ ]:
clear_parquet_files()

In [ ]:
TEST_HAND_FOLDER = os.getenv('TEST_HAND_FOLDER')
COMPLETE_HAND_FOLDER = os.getenv('COMPLETE_HAND_FOLDER')

hands = []
players = []
actions = []

FILE_BATCH_SIZE = 250
file_counter = 0

result = os.walk(COMPLETE_HAND_FOLDER)
file_list = [os.path.join(root, f) for root, dirs, files in result for f in files if f.endswith('.phhs')]
skipped_count = 0
skipped_hands = [] #Keep track of skipped hand ids for later analysis


for curfile in tqdm(file_list):
    with open(curfile, "rb") as f:
        data = tomllib.load(f)

    for hand_id, single_hand_dict in data.items():
        try:
            hand_dict, player_dict, actions_dict = parse_hand(single_hand_dict)
            hands.append(hand_dict)
            players.append(player_dict)
            actions.extend(actions_dict)
        except Exception as e:
            skipped_count += 1
            skipped_hands.append(f"Error processing {curfile} - TOML index {hand_id}: {e}")
            continue #Skip this hand and move on to the next one
    
    #Need to have incremental write down to handle the memory usage of processing such a large dataset. Write to parquet every FILE_BATCH_SIZE files processed.
    file_counter += 1
    if file_counter % FILE_BATCH_SIZE == 0:
        parquet_write_batch(hands, players, actions)
        hands = []
        players = []
        actions = []

#Write any remaining data that hasn't been written yet
if len(hands) > 0: 
    parquet_write_batch(hands, players, actions)